# Pandas Session 4: Capstone Project

## Brazilian E-Commerce Analysis (Olist Dataset)

**Dataset:** Brazilian E-Commerce Public Dataset by Olist

**Source:** Kaggle - https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

---

### Project Overview

This capstone project integrates ALL skills from Sessions 1-3:

- Session 1: Data loading, exploration, indexing, filtering, apply/map
- Session 2: Missing data, cleaning, transformations, merging
- Session 3: GroupBy, aggregation, pivot tables, time series

You will work with a real-world multi-table e-commerce dataset to perform
end-to-end data analysis.

---

### Dataset Description

The Olist dataset contains 100K+ orders from 2016-2018 across multiple tables:

**Tables (8 CSV files):**

1. olist_orders_dataset - Core orders table (order_id, dates, status)
2. olist_order_items_dataset - Items per order (product, price, freight)
3. olist_products_dataset - Product information
4. olist_customers_dataset - Customer information
5. olist_sellers_dataset - Seller information
6. olist_order_payments_dataset - Payment information
7. olist_order_reviews_dataset - Customer reviews
8. product_category_name_translation - Portuguese to English translation

---

### Business Context

You are a data analyst at Olist. The business team needs insights on:

- Customer behavior and segmentation (RFM analysis)
- Product and category performance
- Sales trends and seasonality
- Delivery performance
- Review patterns and customer satisfaction

---

### Grading Rubric

- Part 1: Data Loading and Exploration 
- Part 2: Data Cleaning and Preparation 
- Part 3: Data Integration (Merging) 
- Part 4: Exploratory Analysis 
- Part 5: Time Series Analysis 
- Part 6: Business Insights and Recommendations 


---
## Part 1: Data Loading and Exploration 
---

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Download the dataset from Kaggle:
# https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
# Extract and place CSV files in a 'data/' folder

# Load all datasets
# Adjust paths as needed
data_path = 'data/'  # or your path to the CSV files

# Try loading from local path, provide instructions if not found
try:
    orders = pd.read_csv('olist_orders_dataset.csv')
    order_items = pd.read_csv('olist_order_items_dataset.csv')
    products = pd.read_csv('olist_products_dataset.csv')
    customers = pd.read_csv('olist_customers_dataset.csv')
    sellers = pd.read_csv('olist_sellers_dataset.csv')
    payments = pd.read_csv('olist_order_payments_dataset.csv')
    reviews = pd.read_csv('olist_order_reviews_dataset.csv')
    category_translation = pd.read_csv('product_category_name_translation.csv')
    print('All datasets loaded successfully!')
except FileNotFoundError:
    print('Please download the dataset from Kaggle:')
    print('https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce')
    print('Extract the CSV files to a "data/" folder')

###  1.1: Explore Each Table 

For each of the 8 tables:

1. Show the shape (rows, columns)
2. Display first 3 rows
3. List the column names

Create a summary DataFrame showing table name, row count, and column count.

In [ ]:
tables = {
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'customers': customers,
    'sellers': sellers,
    'payments': payments,
    'reviews': reviews,
    'category_translation': category_translation
}

summary = []

for name, df in tables.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("\nFirst 3 Rows:")
    display(df.head(3))
    print("\nColumns:")
    print(df.columns.tolist())

    summary.append({
        'table_name': name,
        'row_count': df.shape[0],
        'column_count': df.shape[1]
    })

summary_df = pd.DataFrame(summary)

print("\nSummary DataFrame:")
display(summary_df)

###  1.2: Identify Keys and Relationships 

1. Identify the primary key for each table
2. Identify the foreign keys that connect tables
3. Draw a simple text-based diagram showing table relationships

In [ ]:
tables = {
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'customers': customers,
    'sellers': sellers,
    'payments': payments,
    'reviews': reviews,
    'category_translation': category_translation
}

for name, df in tables.items():
    print(f"\n{name.upper()}")
    for col in df.columns:
        if df[col].nunique() == len(df):
            print(f"{col} -> Possible Primary Key")

print("\nForeign Keys")
print("orders.customer_id -> customers.customer_id")
print("order_items.order_id -> orders.order_id")
print("order_items.product_id -> products.product_id")
print("order_items.seller_id -> sellers.seller_id")
print("payments.order_id -> orders.order_id")
print("reviews.order_id -> orders.order_id")
print("products.product_category_name -> category_translation.product_category_name")

print("\nRelationship Diagram")

print("""
customers
    |
customer_id
    |
orders
    |
order_id
    |
+-----------+-----------+
|           |           |
payments   reviews   order_items
                        |
            +-----------+-----------+
            |                       |
       product_id              seller_id
            |                       |
        products                sellers
            |
product_category_name
            |
category_translation
""")

###  1.3: Data Types and Basic Statistics 

For the orders table:

1. Show data types for all columns
2. Identify date columns that need conversion
3. Show value counts for order_status

In [ ]:
print(orders.dtypes)

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

print("\nDate Columns:")
print(date_columns)

print("\nOrder Status Value Counts:")
print(orders['order_status'].value_counts())

---
## Part 2: Data Cleaning and Preparation 
---

###  2.1: Missing Value Analysis 

For each table:

1. Calculate the count and percentage of missing values per column
2. Identify which columns have missing values
3. Document a strategy for handling each

In [ ]:
def analyze_missing(df, name):
    missing_count = df.isnull().sum()
    missing_percent = (df.isnull().sum() / len(df)) * 100

    result = pd.DataFrame({
        'column': df.columns,
        'missing_count': missing_count.values,
        'missing_percentage': missing_percent.values
    })

    result = result[result['missing_count'] > 0]

    print(f"\n{name.upper()}")

    if len(result) > 0:
        display(result.sort_values('missing_percentage', ascending=False))
    else:
        print("No missing values")

tables = {
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'customers': customers,
    'sellers': sellers,
    'payments': payments,
    'reviews': reviews,
    'category_translation': category_translation
}

for name, df in tables.items():
    analyze_missing(df, name)

strategies = pd.DataFrame({
    'Column': [
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'product_category_name',
        'product_name_lenght',
        'product_description_lenght',
        'product_photos_qty',
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm',
        'review_comment_title',
        'review_comment_message'
    ],
    'Strategy': [
        'Keep as NaT',
        'Keep as NaT',
        'Keep as NaT',
        'Fill with Unknown',
        'Median',
        'Median',
        'Median',
        'Median',
        'Median',
        'Median',
        'Median',
        'Fill with No Title',
        'Fill with No Comment'
    ]
})

display(strategies)

###  2.2: Handle Missing Values 

1. In products table: fill missing category names with 'unknown'
2. In orders table: handle missing dates appropriately
3. In reviews table: fill missing review comments with empty string
4. Verify all critical missing values are handled

In [ ]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')

date_columns = [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("Products Missing Values:")
print(products[['product_category_name']].isnull().sum())

print("\nReviews Missing Values:")
print(reviews[['review_comment_title', 'review_comment_message']].isnull().sum())

print("\nOrders Missing Values:")
print(orders[date_columns].isnull().sum())

###  2.3: Date Conversions 

Convert all date columns to datetime:

1. orders: order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date
2. reviews: review_creation_date, review_answer_timestamp

Create derived columns:

- delivery_days: actual delivery time (delivered - purchase)
- estimated_vs_actual: difference between estimated and actual delivery

In [ ]:
date_columns_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns_orders:
    orders[col] = pd.to_datetime(orders[col])

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

orders['delivery_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_purchase_timestamp']
).dt.days

orders['estimated_vs_actual'] = (
    orders['order_estimated_delivery_date']
    - orders['order_delivered_customer_date']
).dt.days

orders[['delivery_days', 'estimated_vs_actual']].head()

###  2.4: Translate Categories 

1. Merge products with category_translation to get English names
2. Handle any categories without translation
3. Create a clean products table with English category names

In [ ]:
products_clean = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english']
    .fillna('unknown')
)

products_clean = products_clean.drop(columns=['product_category_name'])

display(products_clean.head())

print(products_clean.shape)

print(
    products_clean['product_category_name_english']
    .isnull()
    .sum()
)

---
## Part 3: Data Integration - Merging 
---

###  3.1: Create Order Details Table 

Merge orders with order_items to create a detailed order table:

1. Use appropriate join type
2. Verify no data is lost
3. Check the resulting shape

In [ ]:
order_details = orders.merge(
    order_items,
    on='order_id',
    how='left'
)

print("Orders Shape:", orders.shape)
print("Order Items Shape:", order_items.shape)
print("Order Details Shape:", order_details.shape)

print("\nMissing order_id after merge:")
print(order_details['order_id'].isnull().sum())

display(order_details.head())

###  3.2: Create Comprehensive Order Table 

Create a master table by merging:

- orders + order_items
- + products (with English categories)
- + customers
- + sellers

This will be your main analysis table.

In [ ]:
master_table = (
    orders
    .merge(order_items, on='order_id', how='left')
    .merge(products_clean, on='product_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(sellers, on='seller_id', how='left')
)

print("Master Table Shape:", master_table.shape)

display(master_table.head())

print("\nMissing Values:")
print(master_table.isnull().sum().sort_values(ascending=False).head(20))

###  3.3: Add Payment and Review Data 

1. Merge payment information (handle multiple payments per order)
2. Merge review scores (handle orders without reviews)
3. Document your merge strategy and any data handling decisions

In [ ]:
payments_summary = payments.groupby('order_id').agg({
    'payment_sequential': 'max',
    'payment_installments': 'max',
    'payment_value': 'sum'
}).reset_index()

payment_type_mode = (
    payments.groupby('order_id')['payment_type']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .reset_index()
)

payments_summary = payments_summary.merge(
    payment_type_mode,
    on='order_id',
    how='left'
)

reviews_summary = reviews[[
    'order_id',
    'review_score',
    'review_comment_title',
    'review_comment_message'
]]

master_table = (
    master_table
    .merge(payments_summary, on='order_id', how='left')
    .merge(reviews_summary, on='order_id', how='left')
)

print("Master Table Shape:", master_table.shape)

display(master_table.head())

print(master_table[['payment_type', 'payment_value', 'review_score']].isnull().sum())

---
## Part 4: Exploratory Analysis 
---

###  4.1: Category Analysis 

Analyze product categories:

1. Top 10 categories by order count
2. Top 10 categories by total revenue
3. Average order value by category
4. Create a pivot table of categories by customer state

In [ ]:
top_categories_orders = (
    master_table.groupby('product_category_name_english')['order_id']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='order_count')
)

display(top_categories_orders)

top_categories_revenue = (
    master_table.groupby('product_category_name_english')['price']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='total_revenue')
)

display(top_categories_revenue)

avg_order_value = (
    master_table.groupby('product_category_name_english')['price']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='avg_order_value')
)

display(avg_order_value)

category_state_pivot = pd.pivot_table(
    master_table,
    index='product_category_name_english',
    columns='customer_state',
    values='order_id',
    aggfunc='count',
    fill_value=0
)

display(category_state_pivot)

###  4.2: Customer Segmentation - RFM Analysis 

Perform RFM (Recency, Frequency, Monetary) analysis:

For each customer, calculate:

- Recency: Days since last purchase
- Frequency: Number of orders
- Monetary: Total spend

Create RFM scores (1-5) using qcut for each metric.

In [ ]:
reference_date = master_table['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = master_table.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days,
    'order_id': 'nunique',
    'payment_value': 'sum'
}).reset_index()

rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

display(rfm.head())

print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

###  4.3: Seller Performance 

Analyze sellers:

1. Top 10 sellers by revenue
2. Average review score by seller (top 20 by order count)
3. Delivery performance by seller state
4. Which sellers have the fastest delivery times?

In [ ]:
top_10_sellers_revenue = (
    master_table.groupby('seller_id')['payment_value']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='total_revenue')
)

display(top_10_sellers_revenue)

top_20_sellers = (
    master_table.groupby('seller_id')['order_id']
    .nunique()
    .sort_values(ascending=False)
    .head(20)
    .index
)

avg_review_by_seller = (
    master_table[master_table['seller_id'].isin(top_20_sellers)]
    .groupby('seller_id')['review_score']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='avg_review_score')
)

display(avg_review_by_seller)

delivery_by_state = (
    master_table.groupby('seller_state')['delivery_days']
    .mean()
    .sort_values()
    .reset_index(name='avg_delivery_days')
)

display(delivery_by_state)

fastest_sellers = (
    master_table.groupby('seller_id')['delivery_days']
    .mean()
    .sort_values()
    .head(10)
    .reset_index(name='avg_delivery_days')
)

display(fastest_sellers)

###  4.4: Geographic Analysis 

1. Orders by customer state
2. Revenue by customer state
3. Average order value by state
4. Which state pairs (customer-seller) have the most transactions?

In [ ]:
orders_by_state = (
    master_table.groupby('customer_state')['order_id']
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name='order_count')
)

display(orders_by_state)

revenue_by_state = (
    master_table.groupby('customer_state')['payment_value']
    .sum()
    .sort_values(ascending=False)
    .reset_index(name='total_revenue')
)

display(revenue_by_state)

avg_order_value_by_state = (
    master_table.groupby('customer_state')['payment_value']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='avg_order_value')
)

display(avg_order_value_by_state)

state_pairs = (
    master_table.groupby(['customer_state', 'seller_state'])['order_id']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='transaction_count')
)

display(state_pairs.head(20))

---
## Part 5: Time Series Analysis 
---

###  5.1: Sales Trends 

1. Resample orders to monthly frequency - count and revenue
2. Calculate month-over-month growth rate
3. Identify peak months and seasonal patterns
4. Calculate 3-month rolling average of revenue

In [ ]:
master_table['order_purchase_timestamp'] = pd.to_datetime(
    master_table['order_purchase_timestamp'],
    errors='coerce'
)

monthly_sales = (
    master_table
    .drop_duplicates(subset='order_id')
    .groupby(
        master_table['order_purchase_timestamp'].dt.to_period('M')
    )
    .agg({
        'order_id': 'count',
        'payment_value': 'sum'
    })
)

monthly_sales.columns = ['order_count', 'revenue']

monthly_sales['mom_growth_%'] = (
    monthly_sales['revenue'].pct_change() * 100
)

monthly_sales['rolling_3m_revenue'] = (
    monthly_sales['revenue']
    .rolling(window=3)
    .mean()
)

display(monthly_sales)

peak_months = (
    monthly_sales
    .sort_values('revenue', ascending=False)
    .head(10)
)

print("Peak Months")
display(peak_months)

seasonality = (
    master_table
    .drop_duplicates(subset='order_id')
    .assign(
        month=master_table['order_purchase_timestamp'].dt.month
    )
    .groupby('month')
    .agg({
        'order_id': 'count',
        'payment_value': 'sum'
    })
)

display(seasonality)

###  5.2: Day of Week and Hour Analysis 

1. Extract day of week and hour from order timestamp
2. Create a pivot table of orders by day and hour
3. When do customers shop most?
4. Is there a difference in order value by day of week?

In [ ]:
master_table['order_purchase_timestamp'] = pd.to_datetime(
    master_table['order_purchase_timestamp'],
    errors='coerce'
)

orders_time = master_table.drop_duplicates(subset='order_id').copy()

orders_time['day_of_week'] = (
    orders_time['order_purchase_timestamp']
    .dt.day_name()
)

orders_time['hour'] = (
    orders_time['order_purchase_timestamp']
    .dt.hour
)

day_hour_pivot = pd.pivot_table(
    orders_time,
    index='day_of_week',
    columns='hour',
    values='order_id',
    aggfunc='count',
    fill_value=0
)

display(day_hour_pivot)

shopping_peak = (
    orders_time
    .groupby(['day_of_week', 'hour'])['order_id']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='order_count')
)

print("Peak Shopping Times")
display(shopping_peak.head(10))

avg_order_value_day = (
    orders_time
    .groupby('day_of_week')['payment_value']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='avg_order_value')
)

print("Average Order Value by Day")
display(avg_order_value_day)

###  5.3: Delivery Time Analysis 

1. Average delivery time trend over months
2. Delivery time by product category
3. What percentage of orders were delivered late (after estimated date)?
4. Is there correlation between delivery time and review score?

In [ ]:
delivery_trend = (
    master_table
    .dropna(subset=['delivery_days'])
    .assign(
        year_month=master_table['order_purchase_timestamp'].dt.to_period('M')
    )
    .groupby('year_month')['delivery_days']
    .mean()
    .reset_index(name='avg_delivery_days')
)

display(delivery_trend)

delivery_by_category = (
    master_table
    .groupby('product_category_name_english')['delivery_days']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='avg_delivery_days')
)

display(delivery_by_category)

late_orders = master_table[
    master_table['order_delivered_customer_date']
    > master_table['order_estimated_delivery_date']
]

late_percentage = (
    late_orders['order_id'].nunique()
    / master_table['order_id'].nunique()
) * 100

print("Late Delivery Percentage:", round(late_percentage, 2), "%")

delivery_review_corr = (
    master_table[['delivery_days', 'review_score']]
    .corr()
    .iloc[0, 1]
)

print("Correlation Between Delivery Days and Review Score:")
print(delivery_review_corr)

---
## Part 6: Business Insights and Recommendations 
---

###  6.1: Key Performance Indicators 

Create a KPI summary table showing:

- Total Orders
- Total Revenue
- Average Order Value
- Total Customers
- Average Review Score
- On-Time Delivery Rate
- Repeat Customer Rate

In [ ]:
total_orders = master_table['order_id'].nunique()

total_revenue = master_table['payment_value'].sum()

average_order_value = (
    master_table.groupby('order_id')['payment_value']
    .first()
    .mean()
)

total_customers = master_table['customer_unique_id'].nunique()

average_review_score = master_table['review_score'].mean()

on_time_delivery_rate = (
    (
        master_table['order_delivered_customer_date']
        <= master_table['order_estimated_delivery_date']
    ).sum()
    /
    master_table['order_delivered_customer_date'].notna().sum()
) * 100

customer_orders = (
    master_table.groupby('customer_unique_id')['order_id']
    .nunique()
)

repeat_customer_rate = (
    (customer_orders > 1).sum()
    /
    customer_orders.count()
) * 100

kpi_summary = pd.DataFrame({
    'KPI': [
        'Total Orders',
        'Total Revenue',
        'Average Order Value',
        'Total Customers',
        'Average Review Score',
        'On-Time Delivery Rate (%)',
        'Repeat Customer Rate (%)'
    ],
    'Value': [
        total_orders,
        round(total_revenue, 2),
        round(average_order_value, 2),
        total_customers,
        round(average_review_score, 2),
        round(on_time_delivery_rate, 2),
        round(repeat_customer_rate, 2)
    ]
})

display(kpi_summary)

###  6.2: Executive Summary Table 

Create a comprehensive monthly summary with:

- Order count
- Revenue
- Unique customers
- Average order value
- Average review score
- Average delivery time
- Month-over-month changes

In [ ]:
master_table['year_month'] = (
    pd.to_datetime(master_table['order_purchase_timestamp'])
    .dt.to_period('M')
)

executive_summary = (
    master_table
    .drop_duplicates(subset='order_id')
    .groupby('year_month')
    .agg(
        order_count=('order_id', 'count'),
        revenue=('payment_value', 'sum'),
        unique_customers=('customer_unique_id', 'nunique'),
        average_order_value=('payment_value', 'mean'),
        average_review_score=('review_score', 'mean'),
        average_delivery_time=('delivery_days', 'mean')
    )
)

executive_summary['order_count_mom_%'] = (
    executive_summary['order_count']
    .pct_change() * 100
)

executive_summary['revenue_mom_%'] = (
    executive_summary['revenue']
    .pct_change() * 100
)

executive_summary['customers_mom_%'] = (
    executive_summary['unique_customers']
    .pct_change() * 100
)

display(executive_summary)

###  6.3: Business Recommendations 

Based on your analysis, provide 5 actionable recommendations for Olist.
Support each with data from your analysis.

### Your Business Recommendations:

**Recommendation 1:** [Title]

- Finding: 
- Recommendation: 
- Expected Impact: 

**Recommendation 2:** [Title]

- Finding: 
- Recommendation: 
- Expected Impact: 

**Recommendation 3:** [Title]

- Finding: 
- Recommendation: 
- Expected Impact: 

**Recommendation 4:** [Title]

- Finding: 
- Recommendation: 
- Expected Impact: 

**Recommendation 5:** [Title]

- Finding: 
- Recommendation: 
- Expected Impact: 

---
## Project Deliverables Checklist

- [ ] All 8 tables loaded and explored
- [ ] Missing values identified and handled
- [ ] Date columns converted properly
- [ ] Tables merged into analysis-ready format
- [ ] Category, customer, seller, and geographic analysis completed
- [ ] RFM segmentation performed
- [ ] Time series trends analyzed
- [ ] KPIs calculated
- [ ] Business recommendations provided with data support
- [ ] Code is clean, commented, and reproducible

---
